In [1]:
import pandas as pd
import networkx as nx

given_neurons = pd.read_csv("given_neurons.csv")
edge_list = ( # edges are separated by neuropil, but we only care about total synapse count. group by edges and sum synapse counts
    pd.read_csv('connections_princeton.csv')
    .drop(columns=['nt_type'])
    .groupby(['pre_root_id', 'post_root_id'], as_index=False)['syn_count']
    .sum()
)

G = nx.from_pandas_edgelist(edge_list, source='pre_root_id', target='post_root_id', create_using=nx.DiGraph)

## Ignore this cell. Testing purposes

In [38]:
ORN = given_neurons[given_neurons['group'] == 'ORN']['root_id']
# G_rev = G.reverse(copy=False)
dist_from = {}
counter = 1
for orn in ORN:
    print(f"({counter}/{len(ORN)})", end="")
    dist_from[orn] = nx.single_source_shortest_path_length(G, orn)
    counter += 1
    print("\r", end="")

(173/3006)

NodeNotFound: Source 720575941520988343 is not in G

# Shortest paths

### ORN -> OviDN

Ignore this. Execution time is too long. We can analyze OviDN -> corpora allata pathways in the meantime.

In [ ]:
ORN = given_neurons[given_neurons['group'] == 'ORN']['root_id']
OVIDN = given_neurons[given_neurons['group'] == 'OviDN']['root_id']
path_edges = []

iters = len(ORN) * len(OVIDN)
counter = 1
for orn in ORN:
    for ovidn in OVIDN:
        print(f" ({counter}/{iters}) finding pathways between {orn} -> {ovidn}...", end="")
        paths: Generator[list, None, None] = nx.algorithms.all_shortest_paths(G, orn, ovidn)
        print("\r", end="")
        for path in paths:
            # if len(path) > 4: continue # only 3 hops/4 neurons or less are considered as per the original paper's methods
            path_edges.extend(zip(path, path[1:])) # zip(path, path[1:]) is a neat shorthand of generating edges from the list of nodes in the path
            # INTERNEURONS.update(set(path[1:-1])) # each path starts with Ir94e and ends with OviDN, so the interneurons are the nodes in between
        counter += 1

path_edges_df = pd.DataFrame(path_edges, columns=["pre_root_id", "post_root_id"])
print(f"unique path edges discovered: {len(path_edges_df.drop_duplicates())}")

## OviDN -> CA

In [3]:
CA = given_neurons[given_neurons['group'] == 'CA']['root_id']
OVIDN = given_neurons[given_neurons['group'] == 'OviDN']['root_id']
path_edges = []

iters = len(CA) * len(OVIDN)
counter = 1
invalid_counter = 0
total_path_length = 0
min_path_length = 99999
max_path_length = 0
num_paths = 0
for ca in CA:
    for ovidn in OVIDN:
        print(f" ({counter}/{iters}) finding pathways between {ca} -> {ovidn}...", end="")
        try:
            paths: Generator[list, None, None] = nx.algorithms.all_shortest_paths(G, ca, ovidn)
            print("\r", end="")
            for path in paths:
                total_path_length += len(path)
                if len(path) < min_path_length: min_path_length = len(path)
                elif len(path) > max_path_length: max_path_length = len(path)
                num_paths += 1
                path_edges.extend(zip(path, path[1:])) # zip(path, path[1:]) is a neat shorthand of generating edges from the list of nodes in the path
        except Exception as e:
            invalid_counter += 1
            continue
        counter += 1


print()
print(f"no paths: {invalid_counter}")
path_edges_df = pd.DataFrame(path_edges, columns=["pre_root_id", "post_root_id"])
print(f"unique path edges discovered: {len(path_edges_df.drop_duplicates())}")
print(f"shortest path: {min_path_length} | avg path: {total_path_length / num_paths} | longest path: {max_path_length}")

 (17/96) finding pathways between 720575941655645652 -> 720575941417243164...
no paths: 80
unique path edges discovered: 365
shortest path: 4 | avg path: 4.9705159705159705 | longest path: 5


In [41]:
circuit_edges = edge_list.merge(
    path_edges_df.drop_duplicates(),
    on=["pre_root_id", "post_root_id"],
    how="inner",
)

print(circuit_edges.head())
print(circuit_edges.shape)
circuit_edges.to_csv("filtered_edge_list.csv", index=False)

          pre_root_id        post_root_id  syn_count
0  720575941353199152  720575941485447174          3
1  720575941392752132  720575941441244607          5
2  720575941392752132  720575941458019487          4
3  720575941392752132  720575941612233830          3
4  720575941392752132  720575941629729062          3
(1718, 3)


# Unknown neuron identification

In [42]:
import pandas as pd

edge_list = pd.read_csv("filtered_edge_list.csv")
attributes = pd.read_csv("neurons.csv")
given_neurons = pd.read_csv('given_neurons.csv')
givens = set(given_neurons['root_id'])

all_ids = pd.concat([edge_list['pre_root_id'], edge_list['post_root_id']]).drop_duplicates()
unknowns = all_ids[~all_ids.isin(givens)]

attributes = attributes[attributes['Root ID'].isin(unknowns)]
attributes.to_csv("filtered_attributes.csv", index=False)



Briefly describe cell & NT types for unknown neurons

In [43]:
print(attributes.groupby('Primary Cell Type')['Root ID'].count().sort_values(ascending=False))
print()
print(attributes.groupby('Predicted NT type')['Predicted NT confidence'].describe())

Primary Cell Type
INXXX290    7
IN07B061    7
CB1379      7
INXXX446    6
CB1008      4
           ..
CL008       1
CB4242      1
CB4204      1
CB4203      1
s-LNv_a     1
Name: Root ID, Length: 414, dtype: int64

                   count      mean       std   min   25%   50%    75%   max
Predicted NT type                                                          
ACH                163.0  0.708282  0.202678  0.30  0.50  0.76  0.910  0.98
DA                 189.0  0.601429  0.137491  0.31  0.50  0.59  0.710  0.96
GABA               106.0  0.788491  0.135195  0.46  0.71  0.82  0.900  0.97
GLUT               102.0  0.715490  0.164103  0.35  0.59  0.70  0.855  0.98
OCT                  1.0  0.680000       NaN  0.68  0.68  0.68  0.680  0.68
SER                 35.0  0.511143  0.137044  0.37  0.41  0.48  0.580  0.90


Traverse the graph from OviDNs, and for each unknown node mark its hop number

In [44]:
from collections import deque
import networkx as nx

edge_list = pd.read_csv("filtered_edge_list.csv")  # pre_root_id, post_root_id, syn_count
sg = nx.from_pandas_edgelist(
    edge_list, source="pre_root_id", target="post_root_id", create_using=nx.DiGraph
)

OVIDN_set = set(given_neurons[given_neurons['group'] == 'OviDN']['root_id'])
CA_set = set(given_neurons[given_neurons['group'] == 'CA']['root_id'])

hop_number = {}
queue = deque()

for o in OVIDN_set:
    if o in sg:
        hop_number[o] = 0
        queue.append(o)
    else:
        print(f'{o} not in sg')

while queue:
    u = queue.popleft()
    for v in sg.successors(u):
        if v not in hop_number:          # "if not already characterized"
            if v in CA_set: continue
            hop_number[v] = hop_number[u] + 1
            queue.append(v)

In [46]:
import gravis

SER_set = set(attributes[attributes['Predicted NT type'] == "SER"]['Root ID'])
# SER_conf_set = set(attributes[(attributes['Predicted NT type'] == "SER") & (attributes['Predicted NT confidence'] >= 0.5)]['Root ID'])
DA_set = set(attributes[attributes['Predicted NT type'] == "DA"]['Root ID'])

print(SER_set)

for n in sg.nodes():
    if n in OVIDN_set: 
        sg.nodes[n]['color'] = "#a950f2"
    elif n in CA_set: 
        sg.nodes[n]['color'] = "#f25050"
    elif n in SER_set:
        sg.nodes[n]['color'] = "#0000ff"
    elif n in DA_set:
        sg.nodes[n]['color'] = "#FF8800"

        # if n in SER_conf_set:
        #     sg.nodes[n]['color'] = "#0000ff"
        # else:
        #     sg.nodes[n]['color'] = "#809cff"
    else:
        sg.nodes[n]['color'] = "#cecece"
    sg.nodes[n]['hover'] = f"Root ID: {n}"

for u, v in sg.edges():
    # if u in OVIDN_set: 
    #     sg.edges[u, v]['color'] = "#a950f2"
    # elif v in CA_set:
    #     if u in SER_set:
    #         sg.edges[u, v]['color'] = "#0d6400"
    #     else:
    #         sg.edges[u, v]['color'] = "#f25050"
    if u in SER_set:
        # if v in CA_set:
        #     sg.edges[u, v]['color'] = "#f25050"
        # else:
        sg.edges[u, v]['color'] = "#0000ff"
        # if u in SER_conf_set:
        #     sg.edges[u, v]['color'] = "#0000ff"
        # else:
        #     sg.edges[u, v]['color'] = "#809cff"
    # elif u in OVIDN_set and v in SER_set:
    #     sg.edges[u, v]['color'] = "#a950f2"
    elif u in DA_set:
        # if v in CA_set:
        #     sg.edges[u, v]['color'] = "#f25050"
        # else:
        sg.edges[u, v]['color'] = "#FF8800"

    # elif u in OVIDN_set and v in DA_set:
    #     sg.edges[u, v]['color'] = "#a950f2"
    else:
        sg.edges[u, v]['color'] = "#cecece"

# for u, v in sg.edges():
#     if u in SER_set and v in CA_set:
#         sg.nodes[u]['color'] = "#0000ff"
#         sg.edges[u, v]['color'] = "#0000ff"
#     elif u in DA_set and v in CA_set:
#         sg.nodes[u]['color'] = "#FF8800"
#         sg.edges[u, v]['color'] = "#FF8800"
#     else:
#         sg.edges[u, v]['color'] = "#cecece"

gravis.d3(sg, show_node_label=False, node_hover_neighborhood=True, node_hover_tooltip=True, node_label_data_source='label').export_html('network.html')

{720575941545826053, 720575941472733451, 720575941435998094, 720575941560232719, 720575941614563347, 720575941652883477, 720575941393069718, 720575941543199261, 720575941544705821, 720575941540316445, 720575941500002466, 720575941500026530, 720575941595636263, 720575941477200177, 720575941477252913, 720575941567696950, 720575941575361974, 720575941723514426, 720575941634690747, 720575941442233791, 720575941539160525, 720575941492125902, 720575941595083222, 720575941432549847, 720575941467820765, 720575941431472737, 720575941515968099, 720575941612063718, 720575941501625067, 720575941632135916, 720575941632165356, 720575941560951790, 720575941688188399, 720575941568356090, 720575941552961791}


In [ ]:
print(OVIDN_set)

{720575941625363452, 720575941462378692, 720575941596424581, 720575941491649255, 720575941524167755, 720575941626745452, 720575941607634285, 720575941452030330, 720575941561430829, 720575941688560367, 720575941460006128, 720575941644058706, 720575941521454003, 720575941447935764, 720575941614407900, 720575941529824440, 720575941516601690, 720575941417243164}


## Subgraph

In [ ]:
import gravis

sg = G.subgraph(all_ids)

for n in sg.nodes():
    if n in OVIDN_set: 
        sg.nodes[n]['color'] = "#a950f2"
    elif n in CA_set: 
        sg.nodes[n]['color'] = "#f25050"
    sg.nodes[n]['hover'] = f"Root ID: {n}"

for u, v in sg.edges():
    if u in OVIDN_set: 
        sg.edges[u, v]['color'] = "#a950f2"
    elif v in CA_set: 
        sg.edges[u, v]['color'] = "#f25050"
    else:
        sg.edges[u, v]['color'] = "#cecece"

gravis.d3(sg, show_node_label=False, node_hover_neighborhood=True, node_hover_tooltip=True, node_label_data_source='label')

In [ ]:
unreached = [n for n in sg.nodes if n not in hop_number]
print(f"nodes never reached from an OviDN: {len(unreached)}")   # should be 0 if the subgraph is built correctly —
                                                                  # every edge in it came from *some* OviDN→CA path,
                                                                  # so every node should be forward-reachable from an OviDN

ca_hops = {ca: hop_number.get(ca) for ca in CA_set}
print(ca_hops)  # every value should land in [3, 6], matching the min/avg/max you already printed

nodes never reached from an OviDN: 6
{720575941490203081: None, 720575941491151310: None, 720575941450146638: None, 720575941655645652: None, 720575941645177720: None, 720575941506618428: None}


In [ ]:
attributes = pd.read_csv("neurons.csv")
nt_map = attributes.set_index("Root ID")["Predicted NT type"].to_dict()

def label_node(node_id):
    if node_id in OVIDN_set:
        return "OviDN"
    if node_id in CA_set:
        return "CA"
    hop = hop_number.get(node_id, "?")
    nt = nt_map.get(node_id, "Unknown")
    return f"{hop}° {nt} interneuron"

id_to_group = {n: label_node(n) for n in sg.nodes}

print(id_to_group)

def id_to_grp(i):
    return id_to_group.get(i, "Other")

edge_list['pre_group'] = edge_list['pre_root_id'].apply(id_to_grp)
edge_list['post_group'] = edge_list['post_root_id'].apply(id_to_grp)

edge_list.groupby(['pre_group', 'post_group'])['syn_count'].sum().reset_index().to_csv("grouped_edge_list.csv", index=False)

{720575941353199152: '2° DA interneuron', 720575941485447174: '2° DA interneuron', 720575941392752132: '1° ACH interneuron', 720575941441244607: '2° GABA interneuron', 720575941458019487: '2° ACH interneuron', 720575941612233830: '1° ACH interneuron', 720575941629729062: '1° ACH interneuron', 720575941392876548: '2° GABA interneuron', 720575941508254021: '2° ACH interneuron', 720575941392919556: '2° DA interneuron', 720575941506055874: '1° GABA interneuron', 720575941597051761: '1° DA interneuron', 720575941393069718: '3° SER interneuron', 720575941441887067: '2° GLUT interneuron', 720575941451317161: '3° ACH interneuron', 720575941608635245: '2° GLUT interneuron', 720575941393963524: '1° GABA interneuron', 720575941408617519: '2° ACH interneuron', 720575941488395726: '2° GABA interneuron', 720575941668481779: '2° ACH interneuron', 720575941393998852: '2° GABA interneuron', 720575941402070307: '3° DA interneuron', 720575941450001845: '3° DA interneuron', 720575941453425833: '2° DA inte

In [ ]:
import plotly.graph_objects as go

aggregated = pd.read_csv("grouped_edge_list.csv")

# Build global labels
name_to_idx = {}
labels = []

source = []
target = []
value = []

for group in aggregated['pre_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for group in aggregated['post_group'].unique():
    if group not in name_to_idx:
        name_to_idx[group] = len(labels)
        labels.append(group)

for _, row in aggregated.iterrows():
    # if row['pre_group'] == row['post_group']: continue
    source.append(name_to_idx[row['pre_group']])
    target.append(name_to_idx[row['post_group']])
    value.append(row['syn_count'])

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
    ),
    link=dict(
        arrowlen=15,
        source=source,
        target=target,
        value=value,
    ),
)])

fig.show()